In [2]:
import pandas as pd
from xml.dom import minidom
import re
from transformers import AutoTokenizer, RobertaModel, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
import torch

# Reading the Data

In [3]:
platinum_df = pd.read_table('./data/matres/platinum.txt', header=None, sep='\t', names=['docid', 'verb1', 'verb2', 'eiid1', 'eiid2', 'relation'])
# Fixing a typo in the docid
platinum_df.loc[platinum_df['docid'] == 'nyt_20130321_sarcozy', 'docid'] = 'nyt_20130321_sarkozy'
platinum_df['source'] = 'platinum'

aquaint_df = pd.read_table('./data/matres/aquaint.txt', header=None, sep='\t', names=['docid', 'verb1', 'verb2', 'eiid1', 'eiid2', 'relation'])
aquaint_df['source'] = 'aquaint'

relations_df = pd.concat([platinum_df, aquaint_df], ignore_index=True)
relations_df[['eiid1', 'eiid2']] = 'E' + relations_df[['eiid1', 'eiid2']].astype(str)

relations_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation,source
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE,platinum
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE,platinum
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE,platinum
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE,platinum
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE,platinum


In [4]:
# Summary statistics for relation labels
relation_counts = relations_df["relation"].value_counts().sort_index()
relation_percents = relations_df["relation"].value_counts(normalize=True).sort_index().mul(100)

relation_summary = pd.DataFrame({
    "count": relation_counts,
    "percent": relation_percents.round(2)
})

print("Total rows:", len(relations_df))
print("Unique relations:", relations_df["relation"].nunique())
display(relation_summary)

Total rows: 7241
Unique relations: 4


,count,percent
relation,,
AFTER,2532,34.97
BEFORE,3657,50.50
EQUAL,263,3.63
VAGUE,789,10.90


In [5]:
def augment_data(df, verbose=False):
    mask = df["relation"].isin(["BEFORE", "AFTER", "VAGUE", "EQUAL"])
    swapped_df = df.loc[mask].copy()

    # Swap event pair columns
    swapped_df[["verb1", "verb2"]] = swapped_df[["verb2", "verb1"]].to_numpy()
    swapped_df[["eiid1", "eiid2"]] = swapped_df[["eiid2", "eiid1"]].to_numpy()

    # Invert relation
    swapped_df["relation"] = swapped_df["relation"].map({"BEFORE": "AFTER", "AFTER": "BEFORE", "VAGUE": "VAGUE", "EQUAL": "EQUAL"})

    new_df = pd.concat([df, swapped_df], ignore_index=True)

    if verbose:
        print(f"Added {len(swapped_df)} swapped rows")
        print(f"New total rows: {len(new_df)}")

    return new_df

In [6]:
def purify_text(text):
    # Strip TIMEX3 attributes
    text = re.sub(r'<TIMEX3\b[^>]*/>', '[TIMEX3/]', text)
    text = re.sub(r'<TIMEX3\b[^>]*>', '[TIMEX3]', text)
    text = re.sub(r'</TIMEX3>', '[/TIMEX3]', text)

    event_stack = []

    # Replace EVENT tags with their eid values, capitalized
    def _replace_event_tag(match):
        tag = match.group(0)

        if tag.startswith('</EVENT'):
            if event_stack:
                return f"[/{event_stack.pop()}]"
            return tag

        eid_match = re.search(r'\beid\s*=\s*"([^"]+)"', tag)
        if eid_match:
            eid = eid_match.group(1)
            eid = eid.upper()
            event_stack.append(eid)
            return f"[{eid}]"
        return tag

    text = re.sub(r'</EVENT>|<EVENT\b[^>]*>', _replace_event_tag, text)

    return text

def split_sentences(text):
    sentences = re.split(r'(?<!\bMr\.)(?<!\bMs\.)(?<!\bMrs\.)(?<=[.!?])\s+', text)
    return sentences

In [7]:
unique_docids = relations_df['docid'].unique()

doc_texts = {}
for docid in unique_docids:
    filename = f"{docid}.tml"
    source = relations_df.loc[relations_df['docid'] == docid, 'source'].iloc[0]
    folder = 'tempeval' if source == 'platinum' else 'aquaint'
    text_content = minidom.parse(f'./data/{folder}/' + filename).getElementsByTagName('TEXT')[0]
    text_content_str = ''.join(node.toxml() for node in text_content.childNodes).strip()
    text_content_purified = purify_text(text_content_str)
    sentences = split_sentences(text_content_purified)
    doc_texts[docid] = {
        'raw_text': text_content_str,
        'text': text_content_purified,
        'sentences': sentences
    }

docs_df = pd.DataFrame.from_dict(doc_texts, orient='index').reset_index()
docs_df.columns = ['docid', 'raw_text', 'text', 'sentences']
docs_df = docs_df.set_index('docid')
docs_df.head()

,raw_text,text,sentences
docid,,,
WSJ_20130322_159,Israeli Prime Minister Benjamin Netanyahu <EVE...,Israeli Prime Minister Benjamin Netanyahu [E1]...,[Israeli Prime Minister Benjamin Netanyahu [E1...
nyt_20130322_strange_computer,"Our <TIMEX3 type=""DATE"" value=""PRESENT_REF"" ti...",Our [TIMEX3]digital[/TIMEX3] age is all about ...,[Our [TIMEX3]digital[/TIMEX3] age is all about...
CNN_20130321_821,"Barack Obama would <EVENT class=""OCCURRENCE"" e...",Barack Obama would [E1]make[/E1] a great stand...,[Barack Obama would [E1]make[/E1] a great stan...
nyt_20130321_cyprus,"A Cyprus <EVENT class=""OCCURRENCE"" eid=""e2001""...",A Cyprus [E2001]exit[/E2001] from the euro uni...,[A Cyprus [E2001]exit[/E2001] from the euro un...
bbc_20130322_1353,"Israel's prime minister has <EVENT class=""OCCU...",Israel's prime minister has [E1]apologised[/E1...,[Israel's prime minister has [E1]apologised[/E...


# Creating Context

In [8]:
def create_context_window(sentences, eiid1, eiid2, verb1, verb2, padding = 1):
    sentence_indices = []
    for i, sentence in enumerate(sentences):
        if f'[{eiid1}]' in sentence or f'[{eiid2}]' in sentence:
            sentence_indices.append(i)

    if not sentence_indices:
        return ""

    start_index = max(0, min(sentence_indices) - padding)
    end_index = min(len(sentences), max(sentence_indices) + padding + 1)

    context_window = ' '.join(sentences[start_index:end_index])

    context_window = re.sub(rf'\[{re.escape(eiid1)}\]', '[T1]', context_window)
    context_window = re.sub(rf'\[/{re.escape(eiid1)}\]', '[/T1]', context_window)
    context_window = re.sub(rf'\[{re.escape(eiid2)}\]', '[T2]', context_window)
    context_window = re.sub(rf'\[/{re.escape(eiid2)}\]', '[/T2]', context_window)

    # Remove all other event tags like [E3], [/E3], etc.
    context_window = re.sub(r'\[/?E\d+\]', '', context_window)

    # Clean extra whitespace
    context_window = re.sub(r'\s+', ' ', context_window).strip()

    # context_window = f"{verb1} | {verb2} | {context_window}"
    return context_window

In [9]:
relations_df['context_window'] = relations_df.apply(lambda row: create_context_window(docs_df.loc[row['docid'], 'sentences'], row['eiid1'], row['eiid2'], row['verb1'], row['verb2']), axis=1)
relations_df.head()

,docid,verb1,verb2,eiid1,eiid2,relation,source,context_window
0,WSJ_20130322_159,apologized,happened,E1,E5,VAGUE,platinum,Israeli Prime Minister Benjamin Netanyahu [T1]...
1,WSJ_20130322_159,apologized,wrapped,E1,E6,BEFORE,platinum,Israeli Prime Minister Benjamin Netanyahu [T1]...
2,WSJ_20130322_159,apologized,seemed,E1,E10,BEFORE,platinum,Israeli Prime Minister Benjamin Netanyahu [T1]...
3,WSJ_20130322_159,apologized,yield,E1,E11,VAGUE,platinum,Israeli Prime Minister Benjamin Netanyahu [T1]...
4,WSJ_20130322_159,happened,wrapped,E5,E6,BEFORE,platinum,Israeli Prime Minister Benjamin Netanyahu apol...


In [10]:
# Stable mapping: relation label -> integer id
relation2id = {"VAGUE": 0, "BEFORE": 1, "AFTER": 2, "EQUAL": 3}
id2relation = {i: rel for rel, i in relation2id.items()}

num_labels = len(relation2id)

relations_df["relation_id"] = relations_df["relation"].map(relation2id)

In [11]:
# 90% train, 5% validation, 5% test (stratified by label)
train_df, temp_df = train_test_split(
    relations_df,
    test_size=0.1,
    random_state=42,
    stratify=relations_df["relation_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["relation_id"]
)

val_df = augment_data(val_df)
test_df = augment_data(test_df)

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")

Train: 6516
Validation: 724
Test: 726


# Assemble the Model

In [12]:
tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-base")
special = ["[T1]", "[/T1]", "[T2]", "[/T2]", "[TIMEX3]", "[/TIMEX3]"]
tokenizer.add_special_tokens({"additional_special_tokens": special})

6

In [13]:
# special tokens
t1_id = tokenizer.convert_tokens_to_ids("[T1]")
t2_id = tokenizer.convert_tokens_to_ids("[T2]")

In [14]:
def decode_ids(token_ids, skip_special_tokens=False):
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.tolist()
    return tokenizer.decode(token_ids, skip_special_tokens=skip_special_tokens)

def batch_decode_ids(batch_token_ids, skip_special_tokens=False):
    if isinstance(batch_token_ids, torch.Tensor):
        batch_token_ids = batch_token_ids.tolist()
    return tokenizer.batch_decode(batch_token_ids, skip_special_tokens=skip_special_tokens)

In [15]:
class TemporalRelationsModel(torch.nn.Module):
    def __init__(self, num_labels, tokenizer):
        super(TemporalRelationsModel, self).__init__()
        
        config = AutoConfig.from_pretrained("FacebookAI/roberta-base")
        config.is_decoder = False
        self.roberta = RobertaModel.from_pretrained("FacebookAI/roberta-base", config=config)
        self.roberta.resize_token_embeddings(len(tokenizer))

        hidden_size = self.roberta.config.hidden_size
        hidden_dropout_prob = self.roberta.config.hidden_dropout_prob

        self.dropout = torch.nn.Dropout(hidden_dropout_prob)
        self.classifier = torch.nn.Linear(hidden_size * 4, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        # cls_output = self.dropout(outputs.last_hidden_state[:, 0, :])

        hidden = outputs.last_hidden_state
        t1_mask = (input_ids == t1_id)
        t2_mask = (input_ids == t2_id)

        h1 = hidden[t1_mask]
        h2 = hidden[t2_mask]

        pair = torch.cat(
            [h1, h2, h1 - h2, h1 * h2],
            dim=1
        )

        logits = self.classifier(pair)
        return logits

In [16]:
loss_fn = torch.nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)  # (B,) long

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == t1_id).any(dim=1)
        has_t2 = (input_ids == t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]

        optimizer.zero_grad()
        logits = model(input_ids=input_ids, attention_mask=attention_mask)  # (B, C)
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
        all_preds.append(torch.argmax(logits, dim=-1).cpu())
        all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    return total_loss / len(loader.dataset), macro_f1, acc

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0

    all_preds = []
    all_labels = []

    for input_ids, attention_mask, labels in loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == t1_id).any(dim=1)
        has_t2 = (input_ids == t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]

        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(logits, labels)

        preds = torch.argmax(logits, dim=-1)

        total_loss += loss.item() * input_ids.size(0)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()

    macro_f1 = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)

    return {
        "val_loss": total_loss / len(loader.dataset),
        "macro_f1": macro_f1,
        "accuracy": acc,
    }

In [17]:
# Build model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TemporalRelationsModel(num_labels=num_labels, tokenizer=tokenizer).to(device)

# Cross-entropy setup
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To dis

In [18]:
val_encodings = tokenizer(
    val_df["context_window"].tolist(),
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt"
)
val_labels = torch.tensor(val_df["relation_id"].values, dtype=torch.long)
val_dataset = torch.utils.data.TensorDataset(
    val_encodings["input_ids"],
    val_encodings["attention_mask"],
    val_labels
)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=16, shuffle=False)

In [19]:
def create_data_loader(df, tokenizer, batch_size=16):
    encodings = tokenizer(
        df["context_window"].tolist(),
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )
    labels = torch.tensor(df["relation_id"].values, dtype=torch.long)
    dataset = torch.utils.data.TensorDataset(
        encodings["input_ids"],
        encodings["attention_mask"],
        labels
    )

    t1_id = tokenizer.convert_tokens_to_ids("[T1]")
    t2_id = tokenizer.convert_tokens_to_ids("[T2]")

    ids = encodings["input_ids"][0].tolist()
    assert t1_id in ids and t2_id in ids, "Truncated away a target event!"

    print("input_ids shape:", encodings["input_ids"].shape)
    print("attention_mask shape:", encodings["attention_mask"].shape)
    print("labels shape:", labels.shape)
    return torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [20]:
val_loader = create_data_loader(val_df, tokenizer, batch_size=16)
test_loader = create_data_loader(test_df, tokenizer, batch_size=16)

input_ids shape: torch.Size([724, 512])
attention_mask shape: torch.Size([724, 512])
labels shape: torch.Size([724])
input_ids shape: torch.Size([726, 282])
attention_mask shape: torch.Size([726, 282])
labels shape: torch.Size([726])


# First Stage: Basic Data Set

In [21]:
train_loader = create_data_loader(train_df, tokenizer, batch_size=16)

print("num training batches:", len(train_loader))

input_ids shape: torch.Size([6516, 512])
attention_mask shape: torch.Size([6516, 512])
labels shape: torch.Size([6516])
num training batches: 408


In [27]:
# Train
epochs = 12
for epoch in range(epochs):
    train_loss, train_f1, train_acc = train_one_epoch(model, train_loader, optimizer, device)

    val_metrics = evaluate(model, val_loader, device)
    val_loss = val_metrics["val_loss"]
    val_f1 = val_metrics["macro_f1"]
    val_acc = val_metrics["accuracy"]

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

Epoch 1/4 | Train Loss: 0.1177 | Train F1: 0.9064 | Train Acc: 0.9586 | Val Loss: 0.7498 | Val F1: 0.5815 | Val Acc: 0.7784
Epoch 2/4 | Train Loss: 0.0871 | Train F1: 0.9423 | Train Acc: 0.9727 | Val Loss: 0.7552 | Val F1: 0.5816 | Val Acc: 0.7867
Epoch 3/4 | Train Loss: 0.0897 | Train F1: 0.9326 | Train Acc: 0.9696 | Val Loss: 0.7215 | Val F1: 0.6146 | Val Acc: 0.8033
Epoch 4/4 | Train Loss: 0.0644 | Train F1: 0.9609 | Train Acc: 0.9795 | Val Loss: 0.8005 | Val F1: 0.6409 | Val Acc: 0.8061


# Second Stage: Augmented Data Set

In [23]:
augmented_df = augment_data(train_df, verbose=True)
augmented_loader = create_data_loader(augmented_df, tokenizer, batch_size=16)

print("num training batches:", len(augmented_loader))

Added 6516 swapped rows
New total rows: 13032
input_ids shape: torch.Size([13032, 512])
attention_mask shape: torch.Size([13032, 512])
labels shape: torch.Size([13032])
num training batches: 815


In [24]:
# Train
# epochs = 6
# for epoch in range(epochs):
#     train_loss, train_f1, train_acc = train_one_epoch(model, augmented_loader, optimizer, device)

#     val_metrics = evaluate(model, val_loader, device)
#     val_loss = val_metrics["val_loss"]
#     val_f1 = val_metrics["macro_f1"]
#     val_acc = val_metrics["accuracy"]

#     print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}")

# Analysis

In [28]:
from sklearn.metrics import confusion_matrix

# Predict on test set
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for input_ids, attention_mask, labels in test_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        # Keep only samples that contain both target markers
        has_t1 = (input_ids == t1_id).any(dim=1)
        has_t2 = (input_ids == t2_id).any(dim=1)
        valid = has_t1 & has_t2
        if not valid.any():
            continue

        input_ids = input_ids[valid]
        attention_mask = attention_mask[valid]
        labels = labels[valid]
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(logits, dim=-1).cpu()

        all_preds.append(preds)
        all_true.append(labels)

y_pred = torch.cat(all_preds).numpy()
y_true = torch.cat(all_true).detach().cpu().numpy()

# Confusion matrix (rows=true, cols=pred)
label_ids = [0, 1, 2, 3]
cm = confusion_matrix(y_true, y_pred, labels=label_ids)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{id2relation[i]}" for i in label_ids],
    columns=[f"pred_{id2relation[i]}" for i in label_ids],
)
display(cm_df)

test_metrics = evaluate(model, test_loader, device)
print(f"Test Loss: {test_metrics['val_loss']:.4f} | Test F1: {test_metrics['macro_f1']:.4f} | Test Acc: {test_metrics['accuracy']:.4f}")

,pred_VAGUE,pred_BEFORE,pred_AFTER,pred_EQUAL
true_VAGUE,22,32,26,0
true_BEFORE,34,296,16,10
true_AFTER,10,12,222,8
true_EQUAL,2,8,6,10


Test Loss: 0.8556 | Test F1: 0.5898 | Test Acc: 0.7703


# Saving model weights

In [29]:
filename = 'temp_rel_roberta.pt'
torch.save(model.state_dict(), filename)
print(f"Model saved to '{filename}'")

Model saved to 'temp_rel_roberta.pt'
